In [1]:
import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.decomposition import PCA

In [2]:
# Ladataan datasetti
df_raw = pd.read_csv("cicids2017_cleaned.csv")
df100k = df_raw.sample(n=100000, random_state=42)
df = df100k[[
'Init_Win_bytes_forward',
'Flow IAT Mean',
'Packet Length Std',
'Subflow Fwd Bytes',
'Flow Duration',
'Bwd Packet Length Mean',
'Total Length of Fwd Packets',
'PSH Flag Count',
'Flow Packets/s',
'Destination Port',
'Attack Type'
]]

X = df.drop('Attack Type', axis=1)
y = df['Attack Type']

In [3]:
# Splittaus
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


# Pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=0.95)), #Placeholder, korvataan myöhemmin gridissä
   #('poly', PolynomialFeatures(degree=2, include_bias=False)), # TÄMÄ KOMMENTOITU ETTEI AJA KOKO PÄIVÄÄ
    ('logreg', LogisticRegression(class_weight='balanced', max_iter=1000)) # multi_class='multinomial' antoi herjan, että multi_class-parametri poistetaan käytöstä, joten sen jätin pois.
]) 
# kokeiltu max_iter=100 ja max_iter=500, mutta ne eivät riittäneet konvergoitumaan, joten käytin max_iter=1000.

# Parametrit
param_grid = {
    'pca__n_components': [5, 8, 0.95],
    'logreg__C': [0.1, 1, 10]
}

# GridSearchCV
grid_search = GridSearchCV(
    estimator = pipeline, 
    param_grid = param_grid,
    cv=4, 
    scoring='accuracy',
    verbose=1,
    n_jobs=-1
)

# Etsitään parhaat parametrit
grid_search.fit(X_train_full, y_train_full)

# Tulokset
print(f"Parhaat parametrit: {grid_search.best_params_}")
print(f"_________________________________________________________")

# Lopullinen testi parhaalla mallilla
best_model = grid_search.best_estimator_

# TRAINING-RAPORTTI
train_preds = best_model.predict(X_train_full)
print("\n=== Training-raportti (Train Set) ===")
print(classification_report(y_train_full, train_preds))
print(confusion_matrix(y_train_full, train_preds))
print(f"_________________________________________________________")

# VALIDOINTIRAPORTTI
val_preds = cross_val_predict(best_model, X_train_full, y_train_full)
print("\n=== Validointiraportti (Cross-Validation Predictions) ===")
print(classification_report(y_train_full, val_preds))
print(confusion_matrix(y_train_full, val_preds))
print(f"_________________________________________________________")

# LOPULLINEN TESTIRAPORTTI
test_preds = best_model.predict(X_test)
print("\n=== Lopullinen testi (Test Set): ===")
print(classification_report(y_test, test_preds))
print(confusion_matrix(y_test, test_preds))

Fitting 4 folds for each of 9 candidates, totalling 36 fits
Parhaat parametrit: {'logreg__C': 10, 'pca__n_components': 0.95}
_________________________________________________________

=== Training-raportti (Train Set) ===
                precision    recall  f1-score   support

          Bots       0.01      0.63      0.01        70
   Brute Force       0.08      0.96      0.15       302
          DDoS       0.25      0.99      0.40      4092
           DoS       0.70      0.85      0.77      6138
Normal Traffic       0.99      0.59      0.74     66465
 Port Scanning       0.66      0.98      0.79      2859
   Web Attacks       0.04      0.93      0.08        74

      accuracy                           0.64     80000
     macro avg       0.39      0.85      0.42     80000
  weighted avg       0.92      0.64      0.72     80000

[[   44     0     0     0     7    19     0]
 [    0   291     0     0    11     0     0]
 [    0    20  4070     2     0     0     0]
 [    0   255   352  523